# Participant Demographics & Team Diversity (Figure 1)

**Paper:** *Bridging the AI Readiness Gap Among Clinicians: Results from a National,
Interdisciplinary Health Datathon for Trainees*

**Produces:** Figure 1 — a five-panel demographics dashboard:

- **A** Participant backgrounds (degree categories)
- **B** Distribution by year in program
- **C** Top represented institutions
- **D** Primary motivation for joining
- **E** Team composition matrix (institutional vs. training diversity)

## Data

Reads the `Group Dynamic Analysis` sheet of `Manuscript Data (Combined).xlsx` from
`$DATATHON_DATA` (defaults to `../data`). That file contains participant emails and is
**not** included in this repository (PII, IRB-exempt protocol); obtain it from the
corresponding author. See [`../data/README.md`](../data/README.md).

```bash
pip install -r ../requirements.txt
DATATHON_DATA=/path/to/data jupyter notebook demographics_and_diversity.ipynb
```


In [ ]:
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = os.environ.get("DATATHON_DATA", "../data")
XLSX  = os.path.join(DATA_DIR, "Manuscript Data (Combined).xlsx")
SHEET = "Group Dynamic Analysis"

df = pd.read_excel(XLSX, sheet_name=SHEET)
print(f"Total rows loaded: {len(df)}")

# Column 2 is the (unnamed) team-name column in this export.
df.rename(columns={df.columns[2]: "Team Name"}, inplace=True)
df["Email_Normalized"] = df["Email Address"].str.strip().str.lower()
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(by="Timestamp").drop_duplicates(subset=["Email_Normalized"], keep="last")
print(f"Unique registrants after deduplication: {len(df)}")

In [ ]:
# Canonicalize free-text institution names to a closed set.
INSTITUTION_MAP = {
    r'sinai|icahn|mssm|ismms': 'Icahn School of Medicine at Mount Sinai',
    r'utsw|ut\s*southwestern|university of texas southwestern': 'UT Southwestern',
    r'brown|warren\s*alpert': 'Brown University (Warren Alpert)',
    r'loyola|stritch': 'Loyola Stritch School of Medicine',
    r'unc|chapel\s*hill|university of north carolina': 'UNC Chapel Hill',
    r'ucsf|university of california.*san francisco': 'UCSF',
    r'gwu|george\s*washington|gw\s*(school|smhs)': 'George Washington University',
    r'tufts': 'Tufts University',
    r'stanford': 'Stanford University',
    r'emory': 'Emory University',
    r'ucla|david\s*geffen': 'UCLA',
    r'university of miami|miami.*miller': 'University of Miami',
    r'uic|university of illinois': 'University of Illinois',
    r'drexel': 'Drexel University',
    r'northwestern|feinberg': 'Northwestern University',
    r'westernu|western\s*university\s*of\s*health': 'Western University of Health Sciences',
    r'nymc|new\s*york\s*medical\s*college': 'New York Medical College',
    r'rochester': 'University of Rochester',
    r'rush': 'Rush Medical College',
    r'einstein': 'Albert Einstein College of Medicine',
    r'yale': 'Yale School of Medicine',
    r'penn|perelman': 'University of Pennsylvania',
    r'rutgers': 'Rutgers',
    r'hackensack|hmhn': 'Hackensack Meridian School of Medicine',
    r'tulane': 'Tulane University',
    r'duke': 'Duke University',
    r'weill\s*cornell': 'Weill Cornell',
    r'vcu|virginia\s*commonwealth': 'Virginia Commonwealth University',
    r'baylor': 'Baylor College of Medicine',
    r'university of southern california|^usc$': 'University of Southern California',
    r'ucsd': 'UC San Diego',
    r'\buci\b': 'UC Irvine',
    r'fsu|florida\s*state': 'Florida State University',
    r'georgetown': 'Georgetown University',
    r'midwestern': 'Midwestern University',
    r'lecom': 'LECOM',
    r'medical\s*college\s*of\s*georgia': 'Medical College of Georgia',
    r'uab|alabama.*birmingham': 'University of Alabama at Birmingham',
    r'suny\s*downstate': 'SUNY Downstate',
    r'obafemi|oauife': 'Obafemi Awolowo University',
    r'usmanu|danfod': 'Usmanu Danfodiyo University',
    r'carle': 'Carle Illinois College of Medicine',
    r'umkc|missouri.*kansas': 'University of Missouri-Kansas City',
    r'ohsu|oregon\s*health': 'Oregon Health & Science University',
    r'creighton': 'Creighton University',
    r'nyu|grossman': 'NYU Grossman School of Medicine',
    r'stevens\s*institute': 'Stevens Institute of Technology',
    r'texas\s*a&?m|tamu': 'Texas A&M',
    r'pittsburgh|upmc|carnegie\s*mellon': 'University of Pittsburgh',
    r'university of michigan': 'University of Michigan',
    r'university of maryland|umaryland': 'University of Maryland',
    r'nebraska': 'University of Nebraska',
    r'louisville': 'University of Louisville',
    r'cincinnati': 'University of Cincinnati',
    r'colorado|anschutz': 'University of Colorado',
    r'nevada.*reno': 'University of Nevada Reno',
    r'georgia.*tech|ga\s*tech': 'Georgia Institute of Technology',
    r'upstate\s*medical': 'Upstate Medical University',
    r'quinnipiac|netter': 'Quinnipiac University (Netter)',
    r'mcmaster': 'McMaster University',
    r'berkeley': 'UC Berkeley',
    r'nyit': 'NYIT College of Osteopathic Medicine',
    r'wright\s*state': 'Wright State University',
    r'neomed': 'Northeast Ohio Medical University',
    r'geisinger': 'Geisinger Commonwealth',
    r'eastern\s*virginia': 'Eastern Virginia Medical School',
    r'dell\s*medical': 'Dell Medical School',
    r'california\s*northstate': 'California Northstate University',
}

def clean_institution(inst):
    if pd.isna(inst):
        return "Other"
    s = str(inst).lower().strip()
    skip = [r'^n/?a', r'graduated', r'gap\s*year', r'left\s*clinical', r'curanostics',
            r'conduct\s*science', r'freenome', r'uptotrials', r'employee']
    if any(re.search(p, s) for p in skip):
        return "Other / N/A"
    for pattern, name in INSTITUTION_MAP.items():
        if re.search(pattern, s):
            return name
    intl = ['international', 'turkey', 'istanbul', 'brazil', 'porto alegre', 'lagos',
            'pakistan', 'lahore', 'egypt', 'bahamas', 'marmara', 'cayetano',
            'gulf medical', 'sgu', 'caribbean', 'turkische', 'ilorin']
    if any(k in s for k in intl):
        return "Other (International)"
    return "Other"

df["Institution_Category"] = df["Current Institution"].apply(clean_institution)

In [ ]:
def clean_degree(degree):
    if pd.isna(degree):
        return "Unknown"
    d = str(degree).lower().strip()
    if any(x in d for x in ['dual', 'md/phd', 'md/ms', 'md/master', 'mstp']):
        return 'Dual Degree (MD/PhD, MD/MS)'
    if any(x in d for x in ['resident', 'fellow', 'intern', 'pgy']):
        return 'Resident/Fellow'
    if 'phd' in d and 'md' not in d:
        return 'PhD Student'
    if any(x in d for x in ["master", "msc", "mph", "ms ", "m.s."]):
        return "Master's Student"
    if any(x in d for x in ['md', 'do', 'mbbs', 'medical']):
        return 'Medical Student (MD/DO)'
    if any(x in d for x in ['dds', 'dmd', 'dental', 'b.d.s']):
        return 'Dental Student'
    if 'pharmd' in d or 'pharmacist' in d:
        return 'PharmD'
    if any(x in d for x in ['undergraduate', 'bachelor', 'undergrad', '^bs$', 'b.s.']):
        return 'Undergraduate'
    if any(x in d for x in ['research', 'founder', 'n/a', 'gap year']):
        return 'Other / Non-Student'
    return 'Other'

def clean_year(year):
    if pd.isna(year):
        return np.nan
    try:
        return int(str(year).replace('+', '').strip())
    except ValueError:
        return np.nan

df["Degree_Category"] = df["Degree Program(s)"].apply(clean_degree)
df["Year_Clean"] = df["Year in Program"].apply(clean_year)

In [ ]:
# Team-level diversity statistics (members with a valid team assignment).
merged_df = df[(df["Team Name"].notna()) & (df["Team Name"] != "-") &
               (df["Team Name"].astype(str).str.strip() != "")].copy()
print(f"Matched team members: {len(merged_df)} across {merged_df['Team Name'].nunique()} teams")

team_stats = merged_df.groupby("Team Name").agg(
    Team_Size=("Email_Normalized", "count"),
    Unique_Institutions=("Institution_Category", "nunique"),
    Unique_Degrees=("Degree_Category", "nunique"),
    Avg_Year=("Year_Clean", "mean"),
).reset_index()
team_stats.columns = ["Team Name", "Team Size", "Unique Institutions", "Unique Degrees", "Avg Year"]

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.6)
fig = plt.figure(figsize=(22, 20))
gs = fig.add_gridspec(3, 2)

ax1 = fig.add_subplot(gs[0, 0])
degree_counts = df["Degree_Category"].value_counts()
sns.barplot(x=degree_counts.values, y=degree_counts.index, hue=degree_counts.index,
            legend=False, palette="viridis", ax=ax1)
ax1.set_title("Participant Backgrounds", fontweight="bold", fontsize=20)
ax1.set_xlabel("Count", fontsize=18)

ax2 = fig.add_subplot(gs[0, 1])
year_counts = df["Year_Clean"].dropna().value_counts().sort_index()
sns.barplot(x=year_counts.index.astype(int), y=year_counts.values, hue=year_counts.index.astype(int),
            legend=False, palette="rocket", ax=ax2)
ax2.set_title("Distribution by Year in Program", fontweight="bold", fontsize=20)
ax2.set_xlabel("Year", fontsize=18); ax2.set_ylabel("Count", fontsize=18)

ax3 = fig.add_subplot(gs[1, 0])
inst_counts = df["Institution_Category"].value_counts()
# Five institutions tie at n=5 for the final display slots, so a plain head(12) drops one
# at random (row order). Pin the twelve shown in the published figure for a deterministic,
# manuscript-matching panel C.
FIG1C_INSTITUTIONS = [
    "Icahn School of Medicine at Mount Sinai", "UT Southwestern", "Brown University (Warren Alpert)",
    "Loyola Stritch School of Medicine", "University of Illinois", "University of Miami",
    "UNC Chapel Hill", "UCSF", "Emory University", "Tufts University",
    "Western University of Health Sciences", "Usmanu Danfodiyo University",
]
inst_plot = inst_counts.reindex(FIG1C_INSTITUTIONS)
sns.barplot(x=inst_plot.values, y=inst_plot.index, hue=inst_plot.index,
            legend=False, palette="mako", ax=ax3)
ax3.set_title("Top Represented Institutions", fontweight="bold", fontsize=20)
ax3.set_xlabel("Count", fontsize=18)

ax4 = fig.add_subplot(gs[1, 1])
motivation_map = {
    "Wanting to work on a healthcare problem solving team project": "Team Project",
    "A combination of learning AI skills and publication": "Skills & Publication",
    "Learning AI Skills": "Learn AI Skills",
    "Publication": "Publication",
    "Networking": "Networking",
    "Interest": "General Interest",
}
df["Reason_Clean"] = df["What is your primary reason for joining?"].map(motivation_map).fillna("Other")
reason_counts = df["Reason_Clean"].value_counts().head(6)
colors = sns.color_palette("viridis", n_colors=len(reason_counts))
wedges, texts = ax4.pie(reason_counts, colors=colors, startangle=140)
ax4.legend(wedges, [f"{idx} ({val})" for idx, val in reason_counts.items()],
           title="Motivations", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
ax4.set_title("Primary Motivation for Joining", fontweight="bold", fontsize=20)

ax5 = fig.add_subplot(gs[2, :])
matrix_data = pd.crosstab(team_stats["Unique Institutions"], team_stats["Unique Degrees"])
sns.heatmap(matrix_data, annot=True, fmt="d", cmap="mako_r", linewidths=.5,
            cbar_kws={"label": "Number of Teams"}, ax=ax5)
ax5.collections[0].colorbar.set_label("Number of Teams", fontsize=18)
ax5.set_title("Team Composition Matrix: Institutional vs. Training Diversity", fontweight="bold", fontsize=20)
ax5.set_ylabel("Number of Unique Institutions", fontsize=18)
ax5.set_xlabel("Number of Unique Degree Types", fontsize=18)
ax5.invert_yaxis()

for ax, label in zip([ax1, ax2, ax3, ax4, ax5], ["A", "B", "C", "D", "E"]):
    ax.text(-0.1, 1.1, label, transform=ax.transAxes, fontsize=20, fontweight="bold", va="top", ha="right")

plt.tight_layout()
plt.savefig("Figure1.png", dpi=300, bbox_inches="tight")   # publication resolution
plt.savefig("Figure1.pdf", bbox_inches="tight")            # vector
plt.show()